**Imports et création de la session Spark**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import functions as F

# Créer une session Spark
spark = SparkSession.builder \
    .appName("DE2-Lab0-Validation") \
    .master("local[*]") \
    .getOrCreate()

print("Version de Spark:", spark.version)
print("Interface Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 09:57:52 WARN Utils: Your hostname, Wandaogo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 09:57:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 09:57:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version de Spark: 4.0.1
Interface Spark UI: http://10.255.255.254:4040


**2.Lire le CSV avec schéma explicite**

In [5]:
# Définir le schéma
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("category", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("text", StringType(), True),
])

# Lire le CSV - chemin corrigé
df = spark.read.csv("/home/bibawandaogo/Data_Engineering2/Labs/Lab0/data/sample.csv", header=True, schema=schema)
print(f"Nombre de lignes: {df.count()}, Nombre de colonnes: {len(df.columns)}")
df.printSchema()
df.show(5)

Nombre de lignes: 15, Nombre de colonnes: 4
root
 |-- id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- value: double (nullable = true)
 |-- text: string (nullable = true)

+---+--------+-----+--------------------+
| id|category|value|                text|
+---+--------+-----+--------------------+
|  1|    tech| 42.5|distributed syste...|
|  2| science| 88.3|machine learning ...|
|  3|    tech| 15.7|spark shuffle ope...|
|  4|business| 67.2|data warehouses s...|
|  5| science| 93.1|clustering algori...|
+---+--------+-----+--------------------+
only showing top 5 rows


**3. Écrire en Parquet partitionné**

In [6]:
df.write.mode("overwrite") \
    .partitionBy("category") \
    .parquet("outputs/lab0/sample_parquet")

print("Parquet écrit dans outputs/lab0/sample_parquet/")

Parquet écrit dans outputs/lab0/sample_parquet/


**Lire Parquet et comparer les plans d'exécution**

In [7]:
df_parquet = spark.read.parquet("outputs/lab0/sample_parquet")

print("=== Plan de scan CSV ===")
df.explain("formatted")

print("\n=== Plan de scan Parquet ===")
df_parquet.explain("formatted")

=== Plan de scan CSV ===
== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [4]: [id#0, category#1, value#2, text#3]
Batched: false
Location: InMemoryFileIndex [file:/home/bibawandaogo/Data_Engineering2/Labs/Lab0/data/sample.csv]
ReadSchema: struct<id:int,category:string,value:double,text:string>



=== Plan de scan Parquet ===
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [id#35, value#36, text#37, category#38]
Batched: true
Location: InMemoryFileIndex [file:/home/bibawandaogo/Data_Engineering2/Labs/Lab0/outputs/lab0/sample_parquet]
ReadSchema: struct<id:int,value:double,text:string>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [id#35, value#36, text#37, category#38]




**5. Agrégation et capture des métriques**

In [8]:
# Agrégation simple
agg_df = df_parquet.groupBy("category").agg(
    F.count("*").alias("cnt"),
    F.avg("value").alias("avg_value")
)

print("=== Plan d'exécution de l'agrégation ===")
agg_df.explain("formatted")
agg_df.show()

=== Plan d'exécution de l'agrégation ===
== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [value#36, category#38]
Batched: true
Location: InMemoryFileIndex [file:/home/bibawandaogo/Data_Engineering2/Labs/Lab0/outputs/lab0/sample_parquet]
ReadSchema: struct<value:double>

(2) HashAggregate
Input [2]: [value#36, category#38]
Keys [1]: [category#38]
Functions [2]: [partial_count(1), partial_avg(value#36)]
Aggregate Attributes [3]: [count#48L, sum#49, count#50L]
Results [4]: [category#38, count#51L, sum#52, count#53L]

(3) Exchange
Input [4]: [category#38, count#51L, sum#52, count#53L]
Arguments: hashpartitioning(category#38, 200), ENSURE_REQUIREMENTS, [plan_id=88]

(4) HashAggregate
Input [4]: [category#38, count#51L, sum#52, count#53L]
Keys [1]: [category#38]
Functions [2]: [count(1), avg(value#36)]
Aggregate Attributes [2]: [count(1)#45L, avg(value#36)#46]
Results [3

**6.Arrêter la session**

In [9]:
spark.stop()
print("Lab 0 terminé. Environnement validé.")

Lab 0 terminé. Environnement validé.
